In [ ]:
# IMPORTS
from pyspark.sql import SparkSession
import os

In [ ]:
# CREATE SPARK SESSION
spark = SparkSession.builder \
    .appName("Transport Analysis") \
    .getOrCreate()

print("Spark Session Created")

Spark Session Created


In [ ]:
# LOAD DATA (1M rows)

file_path = "../data/processed/final_data.csv"

df = spark.read.csv(
    file_path,
    header=True,
    inferSchema=True
)

print("Data Loaded")

df.show(5)
print("Total Rows:", df.count())

Data Loaded
+---------+-------+-----+-----+------+------------+-------------------+--------------+
|ticket_id|user_id|route|price|  city|vehicle_type|         price_norm|price_category|
+---------+-------+-----+-----+------+------------+-------------------+--------------+
|        1|    948|  D-E|   43| Delhi|       Metro|0.18840579710144928|           Low|
|        2|    591|  C-D|   74| Delhi|       Metro| 0.6376811594202898|        Medium|
|        3|    837|  D-E|   58|Mumbai|       Metro| 0.4057971014492754|        Medium|
|        4|    571|  A-B|   65| Delhi|       Metro| 0.5072463768115942|        Medium|
|        5|    996|  C-D|   90| Delhi|       Metro| 0.8695652173913043|          High|
+---------+-------+-----+-----+------+------------+-------------------+--------------+
only showing top 5 rows

Total Rows: 1000000


In [ ]:
# FILTER DATA
df_filtered = df.filter(df.price > 50)

print("Filtered Data:")
df_filtered.show(5)


# GROUP BY + AGGREGATION
df_grouped = df.groupBy("route").sum("price")

print("Revenue by Route:")
df_grouped.show()


# SELECT SPECIFIC COLUMNS
df_selected = df.select("route", "price")

df_selected.show(5)

Filtered Data:
+---------+-------+-----+-----+-------+------------+-------------------+--------------+
|ticket_id|user_id|route|price|   city|vehicle_type|         price_norm|price_category|
+---------+-------+-----+-----+-------+------------+-------------------+--------------+
|        2|    591|  C-D|   74|  Delhi|       Metro| 0.6376811594202898|        Medium|
|        3|    837|  D-E|   58| Mumbai|       Metro| 0.4057971014492754|        Medium|
|        4|    571|  A-B|   65|  Delhi|       Metro| 0.5072463768115942|        Medium|
|        5|    996|  C-D|   90|  Delhi|       Metro| 0.8695652173913043|          High|
|        6|    788|  B-C|   55|Kolkata|         Bus|0.36231884057971014|        Medium|
+---------+-------+-----+-----+-------+------------+-------------------+--------------+
only showing top 5 rows

Revenue by Route:
+-----+----------+
|route|sum(price)|
+-----+----------+
|  B-C|  16096337|
|  C-D|  16122954|
|  A-B|  16126274|
|  D-E|  16144661|
+-----+----------

In [ ]:
# REGISTER TEMP TABLE
df.createOrReplaceTempView("tickets")

# SQL QUERY
result = spark.sql("""
    SELECT city, SUM(price) AS revenue
    FROM tickets
    GROUP BY city
    ORDER BY revenue DESC
""")

print("Spark SQL Result:")
result.show()

Spark SQL Result:
+-------+--------+
|   city| revenue|
+-------+--------+
|Kolkata|21548455|
| Mumbai|21479308|
|  Delhi|21462463|
+-------+--------+



In [ ]:
# CACHE DATA
df.cache()

print("Data Cached")

# PARTITIONING
df_repartitioned = df.repartition(4)

print("Partitions:", df_repartitioned.rdd.getNumPartitions())

# ACTION AFTER OPTIMIZATION
df.groupBy("vehicle_type").count().show()

Data Cached
Partitions: 4
+------------+------+
|vehicle_type| count|
+------------+------+
|         Bus|333857|
|       Train|333918|
|       Metro|332225|
+------------+------+

